### Theory

**What is it**

bootstrap: method to extimate errors on params fitted  

use it when x and y are not independent, so don't know how to propagate errors  

**How it works**

1. Original Fit  
Perform the fit on your original dataset and record the best-fit parameters 

2.  Generate Resampled Datasets  
Create many (typically 1000 or more) resampled datasets by sampling with replacement from your original data.
Each resampled dataset should be the same size as the original.

3. Repeat the Fit  
For each resampled dataset, perform the fit again and store the resulting parameter values.

4. Compute Parameter Uncertainties  
 For each parameter, compute the standard deviation (or percentiles, for confidence intervals) across all bootstrap fits. This gives an estimate of the uncertainty (standard error) on each parameter.  

params from first try won't be used in the process of finding the uncertainties 

2 methods, depends what i used to fit the first time  
reuse it again, since it's already saved as a function:  

1. if uses the function it must look like, with curve_fit  
find params that give best accord to x and y  

2. used the chi square  
find params that minimize the chi square    

conceptually different, but get same result   

In [12]:
# # If wanted to minimize the chi, it would be

# bootstrap_results = []
# for _ in range(n_bootstrap):
#     indices = np.random.choice(len(x), len(x), replace=True)
#     x_boot = x[indices]
#     y_boot = y[indices]
#     sigma_x_boot = sigma_x[indices]
#     sigma_y_boot = sigma_y[indices]

#     res = opt.minimize(
#         chi2_circle,
#         initial_guess,
#         args=(x_boot, y_boot, sigma_x_boot, sigma_y_boot)
#     )
#     bootstrap_results.append(res.x)

# bootstrap_results = np.array(bootstrap_results)
# errors = np.std(bootstrap_results, axis=0)

### Code

In [13]:
### Data, should have gotten it before, DO NOT COPY IT  

from scipy.optimize import curve_fit  
import numpy as np

def model(x, a, b):
    return a * x + b

x_data = np.array([8.24734, 5.97840, 4.83381, 4.24801, 3.60843, 3.22218, 2.90327, 2.67710, 2.41840, 2.21594])
y_data = np.array([2.31995, 3.92591, 5.10248, 5.64159, 6.19471, 6.77478, 7.22337, 7.09194, 7.71537, 8.13141])  

popt, pcov = curve_fit(model, x_data, y_data)
print("Original fit:", popt)

Original fit: [-0.95208576  9.85409124]


first, do the fit once, to get the popt  

then to the bottstrap, as guesses, pass it the real params computed in the first fit  
this way speed up and the uncertainty gets lower  

In [14]:
def bootstrap(x_data, y_data, model, p):
    '''Find uncertainties on the param fitted'''

    n_bootstrap = 1000
    param_samples = []

    # Bootstrap loop
    for _ in range(n_bootstrap):
        indices = np.random.randint(0, len(x_data), len(x_data))
        
        # to make it work both with pandas and numpy arrays  
        x_sample = x_data.iloc[indices] if hasattr(x_data, 'iloc') else x_data[indices]
        y_sample = y_data.iloc[indices] if hasattr(y_data, 'iloc') else y_data[indices]  
              
        try:
            popt, _ = curve_fit(model, x_sample, y_sample, p0=p)
            param_samples.append(popt)
        except RuntimeError:
            continue  # Skip failed fits
    # so if it's too long, does what it can instead of losing everything  

    param_samples = np.array(param_samples)
    param_errors = np.std(param_samples, axis=0)

    print("Bootstrap parameter errors:", param_errors)
    return param_errors

In [15]:
errors = bootstrap(x_data, y_data, model, popt)

Bootstrap parameter errors: [0.07921759 0.29102259]


In [16]:
# if want other confidence level, not 68%
# in the function of bootstrap, instead of std, use this  
# example: to get 95 % 

# lower = np.percentile(param_samples, 2.5, axis=0)
# upper = np.percentile(param_samples, 97.5, axis=0)
# sigma =(upper-lower)/2